# Stage 0 — Frozen DINOv2 Embedding QC

Thin notebook: it only **imports**, **calls** `src/`, and **displays**. All
Excel/label logic lives in `src/data.py`; all model/embedding logic in
`src/embeddings.py`. Output: `outputs/analysis/qc_embedding_report.csv`.

Fill in the `EDIT-ME` paths in `config.yaml` before running.

In [ ]:
import sys
from pathlib import Path

# Locate the folder root (the dir containing config.yaml) and make src importable.
ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src import data, embeddings

cfg = load_config()
print('model   :', cfg['analysis']['model'])
print('layers  :', cfg['analysis']['layers'])
print('pooling :', cfg['analysis']['pooling'])
print('device  :', cfg['analysis']['device'])
print('output  :', cfg['paths']['output_dir'])

In [ ]:
# Discover figures (prints the parse preview for you to confirm).
figures = data.list_figures(cfg)
figures.head()

In [ ]:
# Optional: label coverage sanity check (skipped cleanly if labels not yet configured).
try:
    labels = data.load_labels(cfg)
    have = figures['patent_id'].isin(labels.index)
    print(f'labelled patents in sheet : {len(labels)}')
    print(f'figures whose patent_id is in the sheet : {int(have.sum())} / {len(figures)}')
except Exception as e:
    print('label load skipped:', type(e).__name__, '-', e)

In [ ]:
# Compute embeddings (or reload if already on disk).
emb_dir = Path(cfg['paths']['output_dir']) / 'embeddings'
if (emb_dir / 'metadata.parquet').exists():
    print('reloading existing embeddings from', emb_dir)
    result = embeddings.load_embeddings(cfg)
else:
    result = embeddings.compute_embeddings(figures, cfg)
    embeddings.save_embeddings(result, cfg)

print('arrays:', sorted(result['arrays'].keys()))

In [ ]:
# Build + persist the Stage-0 QC report.
report = embeddings.qc_report(result, seed=cfg['analysis']['seed'])

csv_path = Path(cfg['paths']['output_dir']) / 'qc_embedding_report.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)
report.to_csv(csv_path, index=False)
print('wrote', csv_path)
report

## Visual inspection only — not a metric

The two histograms below are for eyeballing distribution shape only. They are
**not** Stage-0 pass/fail criteria — the numeric gate is in the report table.

In [ ]:
# Pick a representative (layer, pooling) to visualize.
key = sorted(result['arrays'].keys())[0]
arr = result['arrays'][key]
print('visualizing layer/pooling:', key, 'shape', arr.shape)

sims = embeddings._pairwise_cosine_sample(arr, embeddings.PAIRWISE_SAMPLE_SIZE, cfg['analysis']['seed'])
plt.figure(figsize=(6, 4))
plt.hist(sims, bins=60)
plt.title(f'Pairwise cosine sample — layer{key[0]} {key[1]}\n(visual inspection only — not a metric)')
plt.xlabel('cosine similarity'); plt.ylabel('count')
plt.tight_layout(); plt.show()

In [ ]:
dim_var = arr.var(axis=0)
plt.figure(figsize=(6, 4))
plt.hist(dim_var, bins=60)
plt.title(f'Per-dimension variance — layer{key[0]} {key[1]}\n(visual inspection only — not a metric)')
plt.xlabel('variance'); plt.ylabel('count of dims')
plt.tight_layout(); plt.show()

In [ ]:
# Stage-0 gate.
passed = embeddings.qc_pass_fail(report)
print('=' * 48)
print('STAGE 0:', 'PASS ✅' if passed else 'FAIL ❌')
print('=' * 48)
if not passed:
    bad = report[(report['nan_count'] > 0) | (report['inf_count'] > 0) | (report['all_zero_count'] > 0)]
    print('offending (layer, pooling) rows:')
    display(bad[['layer', 'pooling', 'nan_count', 'inf_count', 'all_zero_count']])